# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f22bdats1e01014-gif/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/f22bdats1e01014-gif/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np

## 1. My lane as an ML task

**Lane:** Refresh / Content Opportunity Scoring

**Task type:** This is primarily a **scoring / ranking** problem, built on top of a
**classification** model. I need to rank pages by review priority, and I generate
that ranking by training a classifier that predicts the probability a page is
declining — then using that probability as the ranking score.

It is not clustering (I'm not grouping pages into unlabeled types) and it is not
pure regression (I don't need an exact numeric value, just a reliable ordering).

## 2. Target or proxy

**Proxy label (starter data):** `is_declining_label = (trend_direction == "down")`

This is a *current-state* proxy, not a true future outcome — it tells us a page is
declining right now, not whether it will decline next. It's a reasonable starting
point because it's cheap and available in the starter dataset, but a stronger
capstone version would use a future-window label instead, e.g.:

`features from prior 90 days -> decline over next 30 days`

I'm using the proxy label for this framing exercise since that's what the starter
data supports, and I'll flag this limitation clearly in Section 5.

## 3. Success metric

**Primary metric: Precision@50**

Why: a content team only has capacity to review a limited number of pages, so what
matters is not overall accuracy but *"of the top 50 pages the system flags, how many
are actually worth reviewing?"* This directly matches the real decision — a reviewer
works through a ranked list, not a random sample.

I'll also track **ROC AUC** and **average precision** as secondary metrics to see
how the ranking holds up beyond just the top 50, but Precision@50 is the number
that ties directly to the real action (limited review capacity).

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one page (one content item)
print("Shape:", df.shape)
print("One row =", "one content item (page), identified by content_id")

df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "trend_direction"]].head(10)

Shape: (30000, 44)
One row = one content item (page), identified by content_id


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,down
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,down
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,down
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,down


The unit of analysis is **one page** (one `content_id`), not a client and not a
day. Each row above represents a single content item with its own visibility,
freshness, and performance signals — exactly the grain the review queue needs,
since a reviewer decides what to do with one page at a time.

In [3]:
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())
print("\nDeclining rate:", round(df["is_declining_label"].mean(), 3))

df[["content_id", "trend_direction", "is_declining_label"]].head(10)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 0.542


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 5. Why ML beats a fixed rule here

A fixed rule (e.g. "stale AND visible") can only combine a couple of signals with
a hand-picked threshold. It can't learn how multiple weaker signals interact —
for example, a page that isn't extremely stale but has a bad CTR *and* a slipping
position might be a stronger review candidate than the rule alone would catch.

This was shown directly in Notebook 1: the hand rule scored ~0.24 Precision@50,
while a trained random forest scored ~0.74 on the same data — roughly 3x better —
because it could weigh many observable signals together instead of relying on
one or two thresholds a human guessed at.

**Caveat:** this is still an observational, in-sample comparison on a 30,000-row
starter slice, using a current-state proxy label. It's evidence the ML approach
is worth pursuing here, not a guarantee it holds at full warehouse scale or under
stricter future-window validation.

## 6. Self-check

- [x] Named the ML task type: scoring/ranking via classification
- [x] Named the target/proxy: `is_declining_label` (current-state proxy; noted
      the future-window limitation)
- [x] Named the success metric: Precision@50, tied to real reviewer capacity
- [x] Showed the unit of analysis as a real dataframe: one row = one page
- [x] Explained why ML beats a fixed rule: multi-signal interaction, evidenced
      by the ~3x Precision@50 lift already observed in Notebook 1
- [x] Tied the output to a real action: a ranked review queue for a content team